# 05 · Models de classificació: qui marxarà?

L'anàlisi de supervivència (notebooks 02–04) respon **quan** marxa un client. Aquí es planteja
la pregunta complementària: **marxarà, sí o no?** És un problema de classificació binària
(1 = baixa, 0 = continua) resolt amb aprenentatge automàtic supervisat.

Es comparen quatre models (regressió logística, arbre de decisió, Random Forest i gradient
boosting) i tres maneres de tractar el desequilibri de classes. Es fan servir **la mateixa neteja i
la mateixa partició train/test** que el model de Cox, de manera que els clients de prova són
exactament els mateixos i els resultats es poden comparar.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
pd.set_option("display.precision", 3)
from src.preparacio import carregar_dades, codificar_cox
from src.cox import ajustar_cox
from src.classificacio import (MODELS, TRACTAMENTS, preparar_dades, ajustar_tots, probabilitats,
                               avaluar, taula_comparativa, corba_roc, corba_guanys, campanya,
                               odds_ratios, estructura_arbre, importancia_variables,
                               frase_metriques, interpretar_or, smote_disponible)
from src.grafics_classificacio import (fig_confusio, fig_roc, fig_odds_ratios, fig_arbre,
                                       fig_importancia, fig_guanys)
from src.grafics import exportar
df = carregar_dades()
dades = preparar_dades(df)

## Preparació de les dades

- **Variables predictives (X):** les 18 característiques del client. Les categòriques es codifiquen
  en columnes 0/1 (*one-hot*) i se n'elimina una categoria de referència, la mateixa que al model
  de Cox (contracte mensual, DSL, xec postal, «No» als serveis).
- **Diferència amb el Cox:** aquí `tenure` (antiguitat) és **una variable predictiva més**; al Cox és
  el temps. `TotalCharges` s'exclou perquè és pràcticament $\text{tenure} \times \text{MonthlyCharges}$.
- Els valors «No internet service» i «No phone service» es converteixen en «No»: la informació ja la
  donen `InternetService` i `PhoneService`, i repetir-la crearia columnes idèntiques.
- **Variable objectiu (y):** `event` (1 = `Churn = Yes`).
- **Partició:** 80 % entrenament i 20 % prova, estratificada per `event` i amb `random_state = 42`
  (funció `dividir_train_test`, compartida amb el Cox).

In [2]:
print("Entrenament:", dades["X_tr"].shape, f"· taxa de baixa {dades['y_tr'].mean():.1%}")
print("Prova:      ", dades["X_te"].shape, f"· taxa de baixa {dades['y_te'].mean():.1%}")
print("\nColumnes:", dades["columnes"])

Entrenament: (5625, 22) · taxa de baixa 26.6%
Prova:       (1407, 22) · taxa de baixa 26.6%

Columnes: ['tenure', 'MonthlyCharges', 'SeniorCitizen', 'Contract_One year', 'Contract_Two year', 'InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'OnlineSecurity_Yes', 'TechSupport_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes', 'PhoneService_Yes', 'MultipleLines_Yes', 'PaperlessBilling_Yes', 'Partner_Yes', 'Dependents_Yes', 'gender_Male']


## Els quatre models

| Model | Com decideix | Com s'interpreta |
|---|---|---|
| **Regressió logística** | $P(\text{baixa}) = \dfrac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \dots + \beta_k x_k)}}$: suma ponderada de les variables convertida en probabilitat | Odds ratio de cada variable, $e^{\beta_j}$ |
| **Arbre de decisió** | Preguntes sí/no encadenades apreses de les dades (profunditat ≤ 4, fulles ≥ 50 clients) | Es llegeix com a regles |
| **Random Forest** | Mitjana de 300 arbres, cadascun amb una mostra a l'atzar de clients i variables | Importància de les variables |
| **Gradient boosting** | Arbres petits en seqüència; cada un corregeix els errors dels anteriors (`HistGradientBoostingClassifier`) | Importància de les variables |

Les dues variables numèriques (`tenure`, `MonthlyCharges`) s'**estandarditzen** dins del *pipeline*:
l'escalat s'ajusta només amb l'entrenament, per evitar fuga d'informació (*data leakage*).

## Desequilibri de classes

Només el 26,6 % dels clients marxen. Es comparen tres tractaments:

- **Cap:** les dades tal com són.
- **Pesos de classe** (`class_weight="balanced"`): els errors amb la classe minoritària pesen més,
  en proporció inversa a la seva freqüència.
- **SMOTE:** es generen clients sintètics de la classe minoritària interpolant entre veïns propers.
  S'utilitza **SMOTENC**, la variant que no inventa valors intermedis a les variables 0/1, i va
  **dins del pipeline**, de manera que només s'aplica a l'entrenament.

Mètriques al conjunt de prova amb llindar 0,5 (lift: contactant el 10 % de clients amb més risc).

In [3]:
# Si SMOTE no es pot carregar en aquest ordinador (vegeu `smote_disponible`), s'omet.
tractaments = [t for t in TRACTAMENTS if t != "SMOTE" or smote_disponible()]
models = {t: ajustar_tots(t, dades) for t in tractaments}
comparativa = pd.concat({t: taula_comparativa(models[t], dades).set_index("model")
                         for t in tractaments}, names=["tractament"])
comparativa

auc  recall  precision     f1  \
tractament      model                                                  
Cap             Regressió logística  0.834   0.578      0.645  0.609   
                Arbre de decisió     0.820   0.500      0.613  0.551   
                Random Forest        0.837   0.481      0.655  0.555   
                Gradient boosting    0.840   0.524      0.658  0.583   
Pesos de classe Regressió logística  0.833   0.783      0.492  0.605   
                Arbre de decisió     0.815   0.816      0.483  0.607   
                Random Forest        0.836   0.789      0.510  0.620   
                Gradient boosting    0.839   0.810      0.500  0.618   
SMOTE           Regressió logística  0.826   0.754      0.501  0.602   
                Arbre de decisió     0.813   0.791      0.489  0.605   
                Random Forest        0.831   0.741      0.537  0.622   
                Gradient boosting    0.831   0.791      0.518  0.626   

                                     exactitud   lift  
tractament      model                                  
Cap             Regressió logística      0.803  2.668  
                Arbre de decisió         0.783  2.615  
                Random Forest            0.795  2.775  
                Gradient boosting        0.801  2.748  
Pesos de classe Regressió logística      0.728  2.615  
                Arbre de decisió         0.719  2.428  
                Random Forest            0.743  2.748  
                Gradient boosting        0.734  2.721  
SMOTE           Regressió logística      0.735  2.508  
                Arbre de decisió         0.725  2.401  
                Random Forest            0.761  2.668  
                Gradient boosting        0.749  2.775

**Lectura.**

- **Sense tractament**, els models només detecten entre el **48 % i el 58 %** de les baixes (recall):
  tendeixen a dir «no marxa», perquè és el que passa la majoria de vegades. L'exactitud és la més
  alta (≈ 80 %), i precisament per això enganya.
- **Amb pesos de classe**, el recall puja fins al **78–82 %** a canvi de baixar la precision
  (≈ 50 %): es detecten moltes més baixes, amb més falses alarmes.
- **SMOTE** dona un resultat semblant als pesos (recall 74–79 %), però és més lent i no millora
  cap mètrica de manera clara.
- **L'AUC gairebé no canvia** entre tractaments (0,81–0,84): el desequilibri no afecta la capacitat
  d'**ordenar** els clients per risc, sinó on cau el llindar de 0,5.
- **Decisió:** es treballa amb **pesos de classe**: tan bo com SMOTE, més senzill i sense crear
  clients artificials.

## Mètriques del millor model

Matriu de confusió del **gradient boosting** amb pesos de classe (millor AUC). Definicions:

- **Recall** $= \dfrac{VP}{VP + FN}$: de les baixes reals, quantes es detecten.
- **Precision** $= \dfrac{VP}{VP + FP}$: dels clients marcats «en risc», quants marxen.
- **F1** $= 2 \cdot \dfrac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$.
- **AUC**: probabilitat que un client que marxa rebi més risc que un que no marxa (0,5 = atzar).

In [4]:
M = models["Pesos de classe"]
probs = {nom: probabilitats(M[nom], dades["X_te"]) for nom in MODELS}
m_gb = avaluar(dades["y_te"], probs["Gradient boosting"], 0.5)
fig = fig_confusio(m_gb, "Matriu de confusió · Gradient boosting (pesos de classe)")
exportar(fig, "cl_confusio"); fig.show()
print(frase_metriques(m_gb).replace("**", ""))

Dels 374 clients de prova que van marxar, el model n'ha detectat 303 (recall 81%) i se n'han escapat 71. Dels 606 clients que ha marcat com a «en risc», 303 van marxar de veritat (precision 50%); els altres 303 són falses alarmes.


### Efecte del llindar

El llindar decideix a partir de quina probabilitat un client es considera «en risc».

In [5]:
pd.DataFrame({ll: avaluar(dades["y_te"], probs["Gradient boosting"], ll) for ll in [0.3, 0.5, 0.7]}).T[
    ["recall", "precision", "f1", "exactitud", "vp", "fp", "fn"]]

,recall,precision,f1,exactitud,vp,fp,fn
0.3,0.906,0.421,0.575,0.643,339.0,467.0,35.0
0.5,0.810,0.500,0.618,0.734,303.0,303.0,71.0
0.7,0.578,0.626,0.601,0.796,216.0,129.0,158.0


**Lectura.** Amb llindar 0,3 es detecta el **91 %** de les baixes, però només el 42 % de les
alertes són encertades; amb 0,7 la precision puja al 63 % però el recall baixa al 58 %. No hi ha
un llindar «correcte»: depèn del cost relatiu de perdre un client (fals negatiu) i de fer una oferta
innecessària (fals positiu). En churn, perdre un client acostuma a ser més car, i això justifica
llindars de 0,5 o inferiors.

## Corbes ROC

In [6]:
rocs = {nom: corba_roc(dades["y_te"], p) for nom, p in probs.items()}
aucs = {nom: avaluar(dades["y_te"], p)["auc"] for nom, p in probs.items()}
fig = fig_roc(rocs, aucs, "Gradient boosting", "Corbes ROC dels quatre models (pesos de classe)")
exportar(fig, "cl_roc"); fig.show()

**Lectura.** Les quatre corbes gairebé es superposen: AUC entre **0,815** (arbre) i **0,839**
(gradient boosting). Els models complexos milloren poc la regressió logística (0,833), cosa que
indica que la relació entre les variables i la baixa és força simple. L'arbre, limitat a 4 nivells
per ser llegible, és el que perd més.

## Per què decideixen així?

### Regressió logística: odds ratios

Odds ratio $= e^{\beta_j}$, mantenint la resta de variables constants. Per a les numèriques s'expressa
per unitat original (+1 mes, +1 €).

In [7]:
ors = odds_ratios(M["Regressió logística"], dades["columnes"])
fig = fig_odds_ratios(ors, "Odds ratios de la regressió logística")
exportar(fig, "cl_odds_ratios"); fig.show()
ors.assign(interpretacio=ors["OR"].map(interpretar_or))[["nom", "OR", "interpretacio"]]

,nom,OR,interpretacio
0,Contracte 2 anys (vs mensual),0.239,Redueix les odds de baixa un 76 %
1,Sense internet (vs DSL),0.383,Redueix les odds de baixa un 62 %
2,Contracte 1 any (vs mensual),0.456,Redueix les odds de baixa un 54 %
3,Seguretat en línia,0.708,Redueix les odds de baixa un 29 %
4,Té telefonia,0.711,Redueix les odds de baixa un 29 %
5,Suport tècnic,0.740,Redueix les odds de baixa un 26 %
6,Té persones a càrrec,0.786,Redueix les odds de baixa un 21 %
7,Transferència automàtica (vs xec postal),0.898,Redueix les odds de baixa un 10 %
8,Còpia de seguretat en línia,0.910,Redueix les odds de baixa un 9 %
9,Té parella,0.958,Redueix les odds de baixa un 4 %


**Lectura.**

- **Contracte:** el de 2 anys redueix les odds de baixa un **76 %** (OR 0,24) i el d'1 any un
  **54 %** (OR 0,46) respecte del mensual. És el factor protector més fort, igual que al Cox.
- **Fibra òptica:** multiplica les odds per **2,9** respecte de la DSL. És el factor de risc més fort.
- **Antiguitat:** OR 0,970 per mes, és a dir, cada any d'antiguitat redueix les odds ≈ 31 %
  ($0{,}970^{12} \approx 0{,}69$).
- **Serveis de suport** (seguretat en línia, suport tècnic) redueixen el risc ≈ 26–29 %; el **xec
  electrònic** l'augmenta (OR 1,36 respecte del xec postal).
- La **quota mensual** surt amb poc efecte (OR 0,99) perquè és pràcticament la suma dels serveis,
  que ja són al model: l'efecte es reparteix entre ells (col·linealitat).
- Limitació: scikit-learn no calcula p-valors per a la regressió logística; la significació
  estadística dels efectes es valora amb el model de Cox (notebook 03).

### Arbre de decisió: les regles

In [8]:
est = estructura_arbre(M["Arbre de decisió"], dades["columnes"], dades["X_tr"], dades["y_tr"])
fig = fig_arbre(est, "Arbre de decisió (profunditat 4, pesos de classe)")
exportar(fig, "cl_arbre"); fig.show()
est[est["fulla"]].sort_values("taxa_real", ascending=False).head(5)[["n", "taxa_real", "prob_model"]]

,n,taxa_real,prob_model
7,824,0.697,0.864
4,588,0.420,0.667
8,871,0.412,0.660
15,176,0.267,0.502
5,802,0.186,0.387


**Lectura.**

- La **primera pregunta** és «contracte de 2 anys?»: és la variable que separa millor els clients.
- La fulla de més risc és **contracte mensual + fibra òptica + antiguitat ≤ 15,5 mesos**: hi ha
  824 clients d'entrenament i en van marxar el **70 %**. És un perfil clar per a campanyes.
- Els clients amb contracte de 2 anys gairebé no marxen (3 %), sigui quina sigui la resta.
- Les probabilitats de les fulles (`prob_model`) són més altes que la taxa real (`taxa_real`)
  perquè el model s'ha entrenat amb pesos de classe, que donen més pes a les baixes. L'ordre de risc
  entre fulles es manté.

### Importància de les variables (els quatre models)

Importància per **permutació** al conjunt de prova: es barregen a l'atzar els valors d'una variable
(totes les seves columnes alhora) i es mesura quant cau l'AUC. És comparable entre models.

In [9]:
importancies = {nom: importancia_variables(M[nom], dades["X_te"], dades["y_te"]) for nom in MODELS}
pd.DataFrame({nom: imp["nom"].head(5).values for nom, imp in importancies.items()},
             index=[f"{i}a" for i in range(1, 6)])

,Regressió logística,Arbre de decisió,Random Forest,Gradient boosting
1a,Antiguitat,Tipus de contracte,Antiguitat,Antiguitat
2a,Servei d'internet,Servei d'internet,Tipus de contracte,Tipus de contracte
3a,Tipus de contracte,Antiguitat,Servei d'internet,Servei d'internet
4a,Quota mensual,Pel·lícules en streaming,Mètode de pagament,Mètode de pagament
5a,Mètode de pagament,Quota mensual,Seguretat en línia,Factura electrònica


In [10]:
fig = fig_importancia(importancies["Gradient boosting"], "Variables més importants · Gradient boosting")
exportar(fig, "cl_importancia"); fig.show()

**Lectura.** Els quatre models coincideixen en les **tres variables clau: antiguitat, tipus de
contracte i servei d'internet**. La resta tenen un pes petit. Aquesta coincidència entre models
molt diferents fa la conclusió robusta.

## De la predicció a l'acció: campanya de retenció

Amb un pressupost limitat, l'empresa ordena els clients de més a menys risc i en contacta els
primers. El **lift** mesura quantes vegades més baixes es troben així que triant clients a l'atzar:

$$\text{lift}(k) = \frac{\text{taxa de baixa entre el } k\,\% \text{ contactat}}{\text{taxa de baixa global}}$$

In [11]:
guanys = {nom: corba_guanys(dades["y_te"], p) for nom, p in probs.items()}
fig = fig_guanys(guanys, "Gradient boosting", 0.1, "Corba de guanys (conjunt de prova)")
exportar(fig, "cl_guanys"); fig.show()
pd.DataFrame({f"{pct:.0%}": {nom: campanya(dades["y_te"], p, pct)["baixes_detectades"]
                             for nom, p in probs.items()} for pct in [0.1, 0.2, 0.3]})

,10%,20%,30%
Regressió logística,98,183,245
Arbre de decisió,91,177,231
Random Forest,103,183,245
Gradient boosting,102,186,245


**Lectura.** Dels 1.407 clients de prova, 374 marxen. Trucant només al **10 %** amb més risc segons
el gradient boosting (141 clients) se'n troben **102** (el 27 % de totes les baixes): **2,7 vegades**
més que a l'atzar, on se'n trobarien ≈ 37. Amb el **20 %** (281 clients) es captura la **meitat** de
les baixes (186). Per prioritzar trucades, les diferències entre models són petites; l'arbre és el
que en troba menys.

## Comparació amb el model de Cox

El Cox (notebook 03) també assigna un risc a cada client. Es compara amb el classificador sobre els
**mateixos clients de prova**: el Cox s'ajusta només amb l'entrenament i es fa servir el seu risc
relatiu $e^{x\beta}$. Cal tenir en compte que el Cox **no inclou l'antiguitat com a variable** (és el
temps), mentre que el classificador sí.

In [12]:
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
cph = ajustar_cox(codificar_cox(dades["train"]))
risc_cox = np.asarray(cph.predict_partial_hazard(codificar_cox(dades["test"]).drop(columns=["duration", "event"])))
p_gb = probs["Gradient boosting"]
k = int(0.2 * len(p_gb))
coincidencia = len(set(np.argsort(-p_gb)[:k]) & set(np.argsort(-risc_cox)[:k])) / k
pd.Series({
    "AUC del risc de Cox (qui marxa)": roc_auc_score(dades["y_te"], risc_cox),
    "AUC del gradient boosting": avaluar(dades["y_te"], p_gb)["auc"],
    "Correlació de rangs (Spearman) Cox vs GB": spearmanr(risc_cox, p_gb)[0],
    "Coincidència del 20 % de més risc": coincidencia,
}).round(3)

AUC del risc de Cox (qui marxa)             0.788
AUC del gradient boosting                   0.839
Correlació de rangs (Spearman) Cox vs GB    0.887
Coincidència del 20 % de més risc           0.665
dtype: float64

**Lectura.**

- Els dos enfocaments **ordenen els clients de manera molt semblant** (correlació de rangs ≈ 0,89) i
  dos terços del 20 % de clients de més risc són els mateixos.
- Per decidir **qui** marxa, el classificador és més precís (AUC 0,84 vs 0,79), sobretot perquè fa
  servir l'antiguitat com a variable.
- Coincideixen en la direcció dels efectes principals (odds ratios i hazard ratios): els contractes
  llargs, no tenir internet, la seguretat en línia i el suport tècnic protegeixen, i la fibra òptica
  augmenta el risc. Difereixen en el **mètode de pagament**: al Cox el que destaca és que el pagament
  automàtic protegeix (el xec electrònic no és significatiu), mentre que a la regressió logística el
  que destaca és que el xec electrònic augmenta el risc.
- **Complementarietat:** el classificador diu **a qui** trucar i la supervivència diu **quan**
  (els primers mesos, notebook 04).

## Conclusions

- Els quatre models assoleixen un **AUC de 0,82–0,84**. El **gradient boosting** és lleugerament el
  millor, però la **regressió logística** gairebé l'iguala i és molt més fàcil d'explicar.
- **El desequilibri de classes s'ha de tractar**: amb pesos de classe el recall passa del ≈ 55 % al
  ≈ 80 %. L'exactitud no és una bona mètrica per a aquest problema.
- **Antiguitat, tipus de contracte i servei d'internet** són les variables decisives en tots els
  models. El perfil de més risc és: contracte mensual, fibra òptica i menys de 16 mesos d'antiguitat.
- Contactant el **10 %** de clients amb més risc es troba el **27 %** de les baixes (lift 2,7).

**Limitacions.**

- El dataset és una fotografia en un sol moment: no permet construir finestres d'observació i de
  predicció. Les variables (serveis, quota) es mesuren al final, també per als clients que ja han
  marxat.
- `tenure` com a variable predictiva: per als clients que marxen és la seva durada total, i per als
  actius la durada fins avui. Això afavoreix el classificador i és una diferència important respecte
  d'un sistema en producció.
- El llindar òptim depèn dels costos de cada error, que no estan disponibles.
- La regressió logística de scikit-learn no dona p-valors.